# 🌱 Lesson 0: Welcome, Setup & the Big Picture
**Phase 2 · Data Preprocessing & Feature Engineering** &nbsp;|&nbsp; ⏱ about 60–75 minutes &nbsp;|&nbsp; 📚 Prerequisite: Phase 1 (Python + exploratory data analysis)

> *Models don't fail because they're weak. They fail because we hand them garbage.*

## 🎯 By the end of this lesson you will be able to
- Explain *why* raw data cannot be fed directly to a model.
- Describe the preprocessing lifecycle and where each lesson fits.
- Verify your Python environment and open the course datasets.
- Refresh the pandas/NumPy skills the rest of the course relies on.

---

## 👋 Welcome

Imagine you are a chef. A customer orders a beautiful dish. Do you throw the vegetables (still muddy, still in the sack) straight into the pan?
Of course not. You **wash, peel, chop, measure and season** first. Chefs call this *mise en place*: "everything in its place".

**Data preprocessing is the mise en place of machine learning.**

In Phase 1 you learned to *look* at data. In Phase 2 you learn to *prepare* it. This is where most real-world machine learning projects are won or lost. Professionals often say that
a simple model on well-prepared data beats a fancy model on messy data, and in this course you will see that with your own eyes, with numbers.

### The three laws of this course
| # | Law | Why it matters |
|---|-----|----------------|
| 1 | **Look before you leap.** Inspect, then transform. | Every cleaning decision should come from evidence, not habit. |
| 2 | **Never let the test set whisper to the training set.** | Data leakage silently produces fantasy results (Lesson 6). |
| 3 | **Document every transformation.** | Future-you (and your colleagues) are strangers who weren't in the room. |

---
## 🗺️ The preprocessing lifecycle (our map for the next two weeks)

```
 RAW DATA ──► 1. CLEAN ──► 2. TRANSFORM ──► 3. ENGINEER ──► 4. SELECT ──► ML-READY DATA
 (messy CSV)   types          scaling          dates           correlation     numeric, complete,
               duplicates     log transform    ratios          statistics      consistent, informative
               missing        encoding         lags/rolling    RFE/importance
               outliers                        seasons         PCA
               invalid
                         ▲                                                     │
                         └──── 5. DON'T LEAK  ·  7. PUT IT ALL IN A PIPELINE ◄──┘
                              (fit on training data only)   (reproducible + safe)
```

## 📅 Course schedule (Weeks 3–4)

| Day | Lesson | What you will do |
|-----|--------|------------------|
| Wk 3 · Mon–Tue | **00** Welcome + **01** Cleaning I | Types, duplicates, missing values |
| Wk 3 · Wed–Thu | **02** Cleaning II | Inconsistent text, invalid values, outliers |
| Wk 3 · Fri | **03** Transformation | Scaling, log, encoding |
| Wk 4 · Mon | **04** Feature engineering | Dates, ratios, rolling windows, lags, seasons |
| Wk 4 · Tue | **05** Feature selection | Correlation, statistics, RFE, importance, PCA |
| Wk 4 · Wed | **06** Data leakage | The most important lesson |
| Wk 4 · Thu | **07** Pipelines | Everything combined, saved and reusable |
| Wk 4 · Fri | **🏆 Project** | Prepare the agricultural dataset for ML |

## 📓 How each lesson works
1. **Concept**: plain-English explanation with an analogy.
2. **Worked example**: on a tiny dataset where you can *see* every number.
3. **Real data**: the same idea on the messy farm dataset.
4. **Try it**: small "pause and predict" prompts.
5. **Problems + hints**: ⭐ to ⭐⭐⭐, solutions in a separate folder.
6. **Quiz + summary**: lock it in.

> 🧠 **Study tip:** run every cell, but *change things*. Break the code on purpose. Replace a number. Ask "what if?" Reading code is not the same as writing it.

---
## 🔧 Step 1: Check your setup
Run the cell below. You want to see versions (roughly) at or above the ones in `requirements.txt`. If an import fails, run `pip install -r requirements.txt` in your terminal.

In [1]:
import sys
print("Python:", sys.version.split()[0])

import numpy, pandas, sklearn, scipy, matplotlib
for lib in (numpy, pandas, sklearn, scipy, matplotlib):
    print(f"{lib.__name__:12s} {lib.__version__}")

Python: 3.12.3


numpy        2.5.4
pandas       3.0.6
sklearn      1.9.1
scipy        1.18.1
matplotlib   3.11.2


Now load the shared helpers. The first cell of **every** notebook looks like this: it adds our `utils/` folder to Python's search path, imports the usual libraries and sets a clean plot style.

In [2]:
import sys
sys.path.append("../utils")            # lets us import course_utils.py and agri_prep.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_utils import *             # load_raw_farm, profile, setup_plots, ...

setup_plots()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
np.random.seed(42)

---
## 🌾 Step 2: Meet the datasets
This course uses three datasets (all **simulated** but built to mimic real East-African agricultural data). Everything is in the `data/` folder, and `data/DATA_DICTIONARY.md` describes every column.

| File | Rows | Used for | What's wrong with it? |
|------|------|----------|-----------------------|
| `farm_records_raw.csv` | ~2,500 | Main dataset: predict **crop yield** (t/ha) | Missing values, duplicates, typos, impossible values, outliers, mixed units |
| `crop_prices_daily.csv` | ~5,500 | Feature engineering with time (lags, rolling) | Gaps, spikes, three crops in one long table |
| `soil_spectra.csv` | 700 | Dimensionality reduction (PCA) | 40 highly correlated columns |

> 🔁 **Using your own Phase 1 data?** Excellent. Every technique transfers. Use the simulated data for the lessons (so that you can compare with our solutions) and your own data in the final project.

Let's look at the star of the show: **raw**, untouched:

In [3]:
raw = load_raw_farm()
print("Shape:", raw.shape)
raw.head(8)

Shape: (2485, 22)


,farm_id,county,soil_type,farm_size_ha,experience,crop,season,planting_date,harvest_date,rainfall_mm,avg_temp_c,soil_ph,nitrogen_ppm,phosphorus_ppm,potassium_ppm,organic_matter_pct,fertilizer_kg_ha,irrigation,pest_pressure,market_price_ksh_kg,total_revenue_ksh,yield_t_ha
0,F0431,Machakos,Clay,1.06,Advanced,Potato,Long rains,2023-04-06,2023-07-18,491.1,21.7,6.61,27.6,11.0,226.4,1.90,66,Yes,High,43.8,174105.0,3.75
1,F0466,Kakamega,Clay,0.77,Advanced,Sorghum,Short rains,2021-11-09,2022-03-13,645.8,19.6,5.83,17.4,18.5,134.6,2.68,NaN,Yes,Medium,53.0,6530.0,0.16
2,F0534,Kisii,Volcanic,1.69,Intermediate,Potato,Long rains,2019-03-16,2019-07-02,905.6,16.6,5.58,18.6,13.4,101.3,2.55,69,Yes,Low,31.8,204220.0,3.80
3,F0281,Kakamega,volcanic,0.56,Intermediate,maize,NaN,2019-03-07,2019-08-13,944.4,20.0,5.10,39.7,13.5,134.5,2.72,0 kg,N,low,53.5,22170.0,0.74
4,F0575,Kakamega,Sandy,2.21,NaN,SORGHUM,Long rains,2020-04-19,2020-08-17,626.3,21.0,NaN,20.3,5.8,90.7,1.40,136,No,Medium,51.4,10223.0,0.09
5,F0064,Nakuru,Sandy,0.79,Expert,Beans,Long rains,2023-03-25,2023-06-24,416.9,18.3,5.30,51.8,7.1,79.4,NaN,0,No,Low,180.4,102612.0,0.72
6,F0541,Trans Nzoia,Loam,0.47,Intermediate,Maize,Long rains,2020-04-08,10 Sep 2020,NaN,18.1,6.35,31.2,23.8,115.4,2.17,57 kg,Yes,Medium,66.3,82577.0,2.65
7,F0007,Kiambu,Loam,1.51,Beginner,Maize,Short rains,2022-11-25,2023-05-03,690.2,18.3,6.91,33.7,8.5,88.7,2.91,NaN,No,High,70.6,205750.0,1.93


Spend 60 seconds looking at that table. What looks suspicious? Write down at least **three** things before reading on.

In [4]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 2485 entries, 0 to 2484
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   farm_id              2485 non-null   str    
 1   county               2485 non-null   str    
 2   soil_type            2410 non-null   str    
 3   farm_size_ha         2436 non-null   float64
 4   experience           2358 non-null   str    
 5   crop                 2485 non-null   str    
 6   season               2435 non-null   str    
 7   planting_date        2485 non-null   str    
 8   harvest_date         2485 non-null   str    
 9   rainfall_mm          2386 non-null   float64
 10  avg_temp_c           2407 non-null   float64
 11  soil_ph              2337 non-null   float64
 12  nitrogen_ppm         2361 non-null   float64
 13  phosphorus_ppm       2361 non-null   float64
 14  potassium_ppm        2363 non-null   float64
 15  organic_matter_pct   2313 non-null   float64
 16 

**What a trained eye notices:**

- `planting_date` and `harvest_date` are `str`/`object`: they are *text*, not dates.
- `fertilizer_kg_ha` is also text, even though it should be a number. Something in there is not a number.
- `irrigation` is text: should be yes/no.
- Many columns have fewer non-null values than the total rows → **missing data**.

Here is our **profiling helper** (you'll use it a lot). It gives one row per column:

In [5]:
profile(raw)

,dtype,n_missing,pct_missing,n_unique,example
farm_id,str,0,0.0,600,F0431
county,str,0,0.0,37,Machakos
soil_type,str,75,3.0,16,Clay
farm_size_ha,float64,49,2.0,298,1.06
experience,str,127,5.1,4,Advanced
crop,str,0,0.0,27,Potato
season,str,50,2.0,2,Long rains
planting_date,str,0,0.0,631,2023-04-06
harvest_date,str,0,0.0,1148,2023-07-18
rainfall_mm,float64,99,4.0,1934,491.1


---
## 🐼 Step 3: Pandas & NumPy mini-refresher
Ten minutes on the exact moves you'll use every day. Run each cell and *predict the output first*.

### 3.1 Selecting

In [6]:
raw["crop"].head(3)                       # one column  -> Series
raw[["crop", "yield_t_ha"]].head(3)       # several     -> DataFrame
raw.loc[0:2, ["county", "rainfall_mm"]]   # rows by LABEL (end is inclusive!)
raw.iloc[0:2, 0:3]                        # rows by POSITION (end is exclusive)

,farm_id,county,soil_type
0,F0431,Machakos,Clay
1,F0466,Kakamega,Clay


### 3.2 Boolean masks (filtering)

In [7]:
wet = raw["rainfall_mm"] > 900          # a True/False Series
print("wet seasons:", wet.sum(), "of", len(raw))
raw.loc[wet, ["county", "rainfall_mm"]].head(3)

wet seasons: 236 of 2485


,county,rainfall_mm
2,Kisii,905.6
3,Kakamega,944.4
18,Kiambu,1020.1


### 3.3 Summaries and group-by

In [8]:
raw[["rainfall_mm", "soil_ph", "yield_t_ha"]].describe().round(2)

,rainfall_mm,soil_ph,yield_t_ha
count,2386.00,2337.00,2447.00
mean,659.08,6.15,2.56
std,191.66,3.41,4.15
min,-317.00,-1.00,-1.97
25%,533.10,5.62,0.49
50%,668.60,6.03,1.50
75%,792.15,6.43,3.07
max,1141.20,99.00,113.30


Look at `soil_ph`: the max is far above 14, which is physically impossible (pH is on a 0–14 scale). Real data always lies a little. That is why we look first.

### 3.4 NumPy in 60 seconds
Pandas columns are built on NumPy arrays. NumPy operations are *vectorised*: they act on whole arrays at once, with no `for` loop.

In [9]:
x = np.array([1, 10, 100, 1000])
print("log10:", np.log10(x))
print("log1p:", np.log1p(x).round(2))               # log(1 + x): safe when x can be 0
print("z-scores:", ((x - x.mean()) / x.std()).round(2))
print("any NaN?", np.isnan(np.array([1.0, np.nan])).any())

log10: [0. 1. 2. 3.]
log1p: [0.69 2.4  4.62 6.91]
z-scores: [-0.66 -0.64 -0.42  1.72]
any NaN? True


> 🧠 **Pause and predict:** `np.nan == np.nan`: True or False?
> <details><summary>Answer</summary>**False!** NaN means "unknown", and an unknown is not equal to another unknown. That's why we use `isna()`, never `== np.nan`.</details>

In [10]:
print(np.nan == np.nan)
print(pd.isna(np.nan))

False
True


### 3.5 One golden pandas habit for this course: **never modify raw data in place**
Always make a copy before changing things, and keep the raw file untouched. If you break something, you can start over.

In [11]:
df = raw.copy()            # work on df, keep raw pristine
df["farm_size_ha"].describe().round(2)

count    2436.00
mean        3.07
std        19.31
min         0.00
25%         0.62
50%         1.07
75%         1.96
max       377.70
Name: farm_size_ha, dtype: float64

You're ready. In the next lesson we start cleaning.

---
## 🧩 Practice problems
Now it is your turn. Each problem is **self-contained** (it tells you what data to load), so you can do them in any order.

**How to work like a pro:**
1. Read the problem twice. Predict the answer *before* you code.
2. Struggle for at least 10 minutes. Struggling is where learning happens.
3. Open a **hint** only when stuck - they get progressively stronger.
4. Compare with `solutions/solutions_00.ipynb` **after** you have your own answer.

Difficulty: ⭐ warm-up · ⭐⭐ core skill · ⭐⭐⭐ challenge

### Problem 0.1 &nbsp; ⭐ &nbsp; First look at the farm data

Load the raw farm records with `load_raw_farm()`. Print (a) the number of rows and columns, and (b) the names of all columns whose dtype is *text* (`object` or `str`).

<details><summary>💡 Hint 1</summary>

`df.shape` gives `(rows, columns)`.

</details>

<details><summary>💡💡 Hint 2</summary>

`df.select_dtypes(include=['object', 'string'])` selects text columns. On new pandas versions text is called `str`; pass `exclude='number'` to be safe.

</details>

In [12]:
# ✏️ Your code here

### Problem 0.2 &nbsp; ⭐ &nbsp; Spot the mess

Use `value_counts()` on the `crop` column of the raw data. How many *distinct spellings* are there? How many *real* crops should there be? (Do not clean anything yet, just observe.)

<details><summary>💡 Hint 1</summary>

`raw['crop'].value_counts()` counts each distinct string.

</details>

<details><summary>💡💡 Hint 2</summary>

`raw['crop'].nunique()` counts distinct values.

</details>

In [13]:
# ✏️ Your code here

### Problem 0.3 &nbsp; ⭐⭐ &nbsp; Filter and create

From the raw data keep only farms with `farm_size_ha` greater than 5. Report how many there are, and add a new column `is_large` (True/False) to the *full* dataframe that marks farms bigger than 5 ha. What percentage of farms are large?

<details><summary>💡 Hint 1</summary>

A boolean mask: `raw['farm_size_ha'] > 5` gives True/False per row.

</details>

<details><summary>💡💡 Hint 2</summary>

The mean of a boolean column is the proportion of True values: `raw['is_large'].mean() * 100`.

</details>

In [14]:
# ✏️ Your code here

### Problem 0.4 &nbsp; ⭐⭐ &nbsp; Group and summarise

Compute the **median** `rainfall_mm` for every `county` in the raw data and sort from wettest to driest. Why might the result look odd? (Look at how many counties you get!)

<details><summary>💡 Hint 1</summary>

`raw.groupby('county')['rainfall_mm'].median()`

</details>

<details><summary>💡💡 Hint 2</summary>

Chain `.sort_values(ascending=False)`. Then count the index: `len(result)`. Should be 10 counties.

</details>

In [15]:
# ✏️ Your code here

---
## ✅ Check your understanding
Answer in your head first, *then* click to reveal.

**Q1.** Which sentence best describes the goal of preprocessing?

- **A)** Making the dataset smaller so training is faster
- **B)** Turning raw, messy data into numeric, consistent, trustworthy inputs for a model
- **C)** Choosing the best machine-learning algorithm
- **D)** Removing every row that contains a missing value

<details><summary>Show answer</summary>

**B**: Preprocessing is about *quality and representation*. Speed can be a side-effect, but never the goal.

</details>

**Q2.** Why do we say 'garbage in, garbage out'?

- **A)** Models are written badly
- **B)** A model can only learn patterns that are present (including wrong ones) in the data it is shown
- **C)** Garbage collection in Python is slow
- **D)** Because clean data always gives 100% accuracy

<details><summary>Show answer</summary>

**B**: Models are pattern-copiers. Show them errors and they faithfully learn the errors.

</details>

---
## 📌 Summary: what to remember
- Preprocessing = turning raw data into trustworthy numeric inputs.
- The lifecycle: clean → transform → engineer → select: all inside a leak-proof pipeline.
- Always inspect first (`info`, `describe`, `profile`, `value_counts`).
- Keep raw data untouched; work on copies.

## 🚧 Common mistakes
- ⚠️ `np.nan == np.nan` is False: use `isna()`.
- ⚠️ `loc` end-points are inclusive; `iloc` end-points are exclusive.

➡️ **Next:** Lesson 01: Data Cleaning I: types, duplicates and missing values.